# 1 Read Probe Data

### Imports

In [ ]:
import pathlib

import matplotlib.pyplot as plt

from amtools.input_output import CaseReader

### Set data path and create case reader

In [ ]:
# define path to example case
example_case_path = (pathlib.Path.cwd() / "example_case").resolve()

# create case reader object
case = CaseReader(example_case_path)

### Read in probe data & do some analysis

In [ ]:
# read in the probe data for the field velocity (U)
# "combined" will read in all the available checkpoints and combine
# them into a single reader object
U_probe = case.probe("example_probes", "U", "combined")

# read in the probes of the pressure field (p)
# "first" will read in the first checkpoint. Other options exists:
# - "closest to"
# - "exactly"
# - "latest"
p_probe = case.probe("example_probes", "p", "first")

# get the time
U_time = U_probe.time
p_time = p_probe.time

# we can see how the data from different check points has been combined
print(f"The combined U_time: {U_time}")
print(f"The first only p_time: {p_time}\n")

# the data of the probes can be accessed using the data member
U_data = U_probe.data
p_data = p_probe.data

# This will have shape [N_timesteps, N_probes, N_field]
print(
    f"U_data has {U_data.shape[0]} timesteps, {U_data.shape[1]} probes, and {U_data.shape[2]} components (x,y,z)"
)

# note that since p is a scalar field, it has shape [N_timesteps, N_probes]
print(f"p_data has {p_data.shape[0]} timesteps, {p_data.shape[1]} probes")

# we can add an empty dimension using numpy
p_data = p_data[:, :, None]  # or p_data[:,:,np.newaxis]
print(
    f"now p_data has {p_data.shape[0]} timesteps, {p_data.shape[1]} probes, and {p_data.shape[2]} components\n"
)

# The reader also stores the positions of the probes
x = U_probe.x
y = U_probe.y
z = U_probe.z

print(f"There are {len(x)} probes with positions:")
for i, _ in enumerate(x):
    print(f"\tProbe {i}: ({x[i]}, {y[i]}, {z[i]})")
print()

# There is a helper function used to find the nearest probe
target_position = (4.1, 5.1, 6.1)
p = U_probe.find_nearest_probe(target_position)
print(f"The nearest probe to {target_position} is the probe with index {p}")

# This also allows for searching along a single axis
target_y = 7.9
p = U_probe.find_nearest_probe(target_y, dimension="y")
# or
p = U_probe.find_nearest_probe(target_y, dimension=1)
print(f"The nearest probe to y = {target_y} is the probe with index {p}\n")

# We can crop the time by an upper and lower limit
print(f"The probe sample times are {U_probe.time}")

upper_limit = 0.4
lower_limit = 0.1
U_probe.crop_by_time(lower_limit=lower_limit, upper_limit=upper_limit)
print(f"Now the probe sample times are {U_probe.time}\n")
# other ways of cropping the data exist (i.e. by probe, by component)

# We can use np for computing the mean and variance
U_mean = U_probe.data.mean(axis=0)  # axis=0 ensures we average over the time dimension
U_var = U_probe.data.var(axis=0)

### Plotting Examples

We can use the reader to easily plot some quantities. For example, the velocity components of probe 1 against time

In [ ]:
plt.figure()
plt.plot(U_probe.time, U_probe.data[:, 1, :])
plt.title("Velocity Components of Probe 1 Against Time")
plt.xlabel("time")
plt.ylabel("vel")

Plot the $x$-velocity of all probes against time

In [ ]:
plt.figure()
plt.plot(U_probe.time, U_probe.data[:, :, 0])
plt.title("x-velocity Component of All Probes Against Time")
plt.xlabel("time")
plt.ylabel("vel")

or the time-averaged $x$ component of velocity against probe $z$-position

In [ ]:
plt.figure()
plt.plot(U_probe.z, U_probe.data[:, :, 0].mean(axis=0))
plt.title("Time-Averaged x-velocity of All Probes Against z-Position")
plt.xlabel("z")
plt.ylabel("time-averaged vel")